In [33]:
import numpy as np

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd
from metrics import metrics_reg
import time

In [2]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")
X_train = train_data["x"]
X_val = val_data["x"]
X_test = test_data["x"]
y_train = train_data["age"]
y_val = val_data["age"]
y_test = test_data["age"]

train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")
X_train_s = train_data_s["x"]
X_val_s = val_data_s["x"]
X_test_s = test_data_s["x"]
y_train_s = train_data_s["age"]
y_val_s = val_data_s["age"]
y_test_s = test_data_s["age"]

scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]


# with open("data_processed/dataframes/input_cols_pruned.pkl", "rb") as f:
#     INPUT_COLS_PRUNED = pickle.load(f)

# feature_names = list(INPUT_COLS_PRUNED)

In [32]:
l = 11
metrics_df = pd.read_csv(f"Results/test_metrics/{l}layers/test_metrics_{l}layers.csv")
hist = pd.read_csv(f"Results/train_history/{l}layers/train_history_{l}layers.csv")
time = hist["time"].to_numpy().mean()
baseline = {"R2": metrics_df["R2"].values[0], 
            "RMSE": metrics_df["RMSE"].values[0], 
            "MAE": metrics_df["MAE"].values[0],
            "time": time}

In [35]:
def fit_evaluate_pcr(
    X_train,
    y_train,
    X_test,
    y_test,
    std,
    avg,
    n_components=20,
):
    """
    Principal Component Regression for age prediction.

    Parameters
    ----------
    X_train, X_test : array-like, shape (n_samples, n_features)
        Training and test features.

    y_train, y_test : array-like, shape (n_samples,)
        Age labels.

    n_components : int
        Number of PCA components to keep.

    Returns
    -------
    model : sklearn Pipeline
        Fitted PCR model.

    metrics : dict
        Test R2, RMSE, and MAE.
    """

    pcr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components)),
        ("regression", LinearRegression())
    ])
    start = time.time()
    # Fit PCA + regression only on training data
    pcr_model.fit(X_train, y_train)

    end = time.time()
    # Predict on test data
    out = pcr_model.predict(X_test)   
    
    y_pred = (out * std + avg).ravel()
    r2, rmse, mae = metrics_reg(y_test.ravel(), y_pred)
    
    # r2 = r2_score(y_test, out)
    # rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # mae = mean_absolute_error(y_test, y_pred)
    # rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # mae = mean_absolute_error(y_test, y_pred)

    metrics = {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "time": end-start
    }

    return pcr_model, metrics, y_pred

In [36]:
pcr_model, metrics, y_pred = fit_evaluate_pcr(
    X_train=X_train_s,
    y_train=y_train_s,
    X_test=X_test_s,
    y_test=y_test,
    std=y_std,
    avg=y_mean,
    n_components=18,
)

print(metrics)

for key in metrics.keys():
    drop = round(100*(baseline[key]-metrics[key]) / baseline[key], 2)
    print(f"{key}: {abs(drop)}%")

{'R2': 0.5544363856315613, 'RMSE': 9.103636515972143, 'MAE': 7.189499855041504, 'time': 0.021952152252197266}
R2: 20.6%
RMSE: 21.52%
MAE: 22.79%
time: 99.97%


In [37]:
pcr_model, metrics, y_pred = fit_evaluate_pcr(
    X_train=X_train_s,
    y_train=y_train_s,
    X_test=X_test_s,
    y_test=y_test,
    std=y_std,
    avg=y_mean,
    n_components=25,
)

print(metrics)

for key in metrics.keys():
    drop = round(100*(baseline[key]-metrics[key]) / baseline[key], 2)
    print(f"{key}: {abs(drop)}%")

{'R2': 0.5714292526245117, 'RMSE': 8.928352423025544, 'MAE': 7.061859607696533, 'time': 0.1351475715637207}
R2: 18.16%
RMSE: 19.18%
MAE: 20.61%
time: 99.84%
